# Basic Validation Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/01_quickstart/basic_validation/tutorial.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/01_quickstart/basic_validation/tutorial.ipynb)

## Business Scenario

Every team starts with a messy CSV: customer signups, leads, or support tickets. When bad rows enter your lakehouse, every downstream report becomes unreliable. You need a fast, repeatable way to validate data before it lands.

Common problems:
- Missing emails or invalid formats
- Negative or impossible values
- Duplicate records

## Value Proposition

- Contract-driven validation: define schema and rules once in YAML
- Quarantine by default: bad rows are isolated with clear reasons
- Consistent results: same rules run locally, in CI, or in production

---

## Goals

1. Load a raw CSV without manual parsing
2. Enforce schema and quality rules
3. Separate good rows from quarantined rows


## 1. Setup
First, we'll install LakeLogic and import the necessary libraries.

Note: We no longer need to import `polars` or `pandas` directly for data loading; LakeLogic can handle it using the engine specified in your settings or code.

In [16]:
# %pip install lakelogic[all]
import os
from lakelogic import DataProcessor

# You can set the engine via environment variable for global consistency
os.environ["LAKELOGIC_ENGINE"] = "spark"

## 2. Execute directly from Source
We have a CSV with some 'dirty' data. Instead of loading it manually, we call `run_source()` which uses the engine's optimized native readers.

In [17]:
processor = DataProcessor(contract="contract.yaml")
print("Engine:", processor.engine_name)
df_raw, df_good, df_bad = processor.run_source("data/sample_customers.csv")


2026-02-08 00:22:55.687 | INFO     | lakelogic.core.processor:run_source:371 - Loading source: D:\Github\_SaaS\lakelogic\examples\01_quickstart\basic_validation\data\sample_customers.csv via spark


Engine: spark


2026-02-08 00:23:06.456 | INFO     | lakelogic.core.processor:run:276 - Starting LakeLogic run [Auto-Engine: spark, Contract: Tutorial Contract]
2026-02-08 00:23:09.623 | INFO     | lakelogic.core.processor:run:295 - Run complete. Source: 6, Total (post-transform): 5, Good: 3, Quarantined: 2, Pre-Transform Dropped: 1, Ratio: 40.00%


## 3. Inspect the Results
Let's see the raw data, what made it to our 'Silver' layer and what was sent to 'Quarantine'.

In [18]:
print("✅ RAW DATA:")

if os.environ["LAKELOGIC_ENGINE"] == "spark":
    df_raw.show(100, False)
else:
    display(df_raw)

✅ RAW DATA:
+---+-------+---------------------+---+-----------+--------+
|id |name   |email                |age|signup_date|status  |
+---+-------+---------------------+---+-----------+--------+
|1  |Alice  |alice@example.com    |25 |2023-01-01 |active  |
|2  |Bob    |bob@example.com      |30 |2023-01-02 |active  |
|3  |Charlie|charlie-invalid-email|35 |2023-01-03 |active  |
|4  |David  |david@example.com    |-5 |2023-01-04 |inactive|
|5  |Eve    |eve@example.com      |28 |2023-01-05 |active  |
|6  |Alice  |alice@example.com    |25 |2023-01-06 |active  |
+---+-------+---------------------+---+-----------+--------+



In [19]:
print("✅ GOOD DATA (Silver):")

if os.environ["LAKELOGIC_ENGINE"] == "spark":
    df_good.show(100, False)
else:
    display(df_good)

✅ GOOD DATA (Silver):
+---+-----+-----------------+---+-----------+------+----------+
|id |name |email            |age|signup_date|status|is_premium|
+---+-----+-----------------+---+-----------+------+----------+
|6  |Alice|alice@example.com|25 |2023-01-06 |active|false     |
|2  |Bob  |bob@example.com  |30 |2023-01-02 |active|true      |
|5  |Eve  |eve@example.com  |28 |2023-01-05 |active|true      |
+---+-----+-----------------+---+-----------+------+----------+



In [20]:
print("\n🛑 BAD DATA (Quarantine):")
if os.environ["LAKELOGIC_ENGINE"] == "spark":
    df_bad.show(100, False)
else:
    display(df_bad)


🛑 BAD DATA (Quarantine):
+---+-------+---------------------+---+-----------+--------+--------------------------------------------------------------------------------+--------------------------+----------------+----------------------+
|id |name   |email                |age|signup_date|status  |_lakelogic_errors                                                               |_lakelogic_categories     |quarantine_state|quarantine_reprocessed|
+---+-------+---------------------+---+-----------+--------+--------------------------------------------------------------------------------+--------------------------+----------------+----------------------+
|3  |Charlie|charlie-invalid-email|35 |2023-01-03 |active  |[Rule failed: Valid Email (email LIKE '%@%')]                                   |[correctness]             |active          |false                 |
|4  |David  |david@example.com    |-5 |2023-01-04 |inactive|[Rule failed: Valid Age (age > 0), Rule failed: Active Only (status = 'active'

## 4. Analysis
Notice the following:
- **Deduplication**: One of the 'Alice' records was dropped (the older one by signup_date).
- **Schema Failure**: 'Charlie' was quarantined due to an invalid email format.
- **Rule Failure**: 'David' was quarantined because his age was -5 and status was 'inactive'.
- **Enrichment**: The `is_premium` column was correctly derived for the good data.
